# 04 · Validate — conformational specificity (fibril vs monomer) + head-to-head

**Standard slot:** *validate (in silico).* **For Project 11 this is the HARD PART:** the
**conformational-specificity test** — model each survivor against the **monomer** and the **fibril**
and require it to **prefer the fibril** (`specificity_gap > 0`) — plus the BindCraft-vs-RFdiffusion
head-to-head and the **cross-amyloid** specificity extension (tau vs α-syn), with publication-style
figures (D3 part 2).

Needs `results/bindcraft_designs.csv` + `results/rfdiffusion_designs.csv` + `results/all_ranked.csv`
(from notebooks 02–03).

> **Why this notebook matters most.** A binder that scores beautifully on the fibril but ALSO binds the
> abundant monomer is useless as a fibril-specific tracer. Selectivity — not raw fibril affinity — is
> what makes this a diagnostic. Expect **most** designs to fail the monomer counter-test; report that
> honestly.

## Setup paths

In [ ]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## 1 · The conformational-specificity test (fibril vs monomer)

For every design we score **both** conformers with `conformational_specificity()` and compute
`specificity_gap = pae_monomer − pae_fibril`. **Positive & large ⇒ prefers the fibril** (what we
want). We regenerate the pools deterministically (mock) so this notebook is self-contained, then run
the two-state evaluation.

**Caveat, stated up front:** the monomer is intrinsically disordered, so its model (and therefore the
gap) carries extra uncertainty. The gap is a teaching proxy on a model metric — **not** a measured
fold-selectivity. The wet-lab fibril-vs-monomer assay (notebook 05) is what actually proves it.

In [ ]:
import pandas as pd, numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import binder_tools as bt

TARGET, HOTSPOTS = "TAU_PHF", bt.parse_hotspots("A306,A310,A315,A320")

# Rebuild the pools deterministically (mock) and run the TWO-STATE specificity evaluation.
bc = bt.generate_binders_bindcraft(TARGET, HOTSPOTS, n=60, tool="mock");  bt.score_designs(bc, tool="mock")
rf = bt.generate_binders_rfdiffusion(TARGET, HOTSPOTS, n=200, tool="mock"); bt.score_designs(rf, tool="mock")
bt.evaluate_specificity(bc, tool="mock")     # fills pae_fibril / pae_monomer / specificity_gap
bt.evaluate_specificity(rf, tool="mock")

def spec_df(designs, label):
    return pd.DataFrame([dict(
        design_id=d.design_id, paradigm=label, length=d.length,
        pae_fibril=d.pae_fibril, pae_monomer=d.pae_monomer,
        specificity_gap=d.specificity_gap, sequence=d.sequence,
        hotspot_overlap=bt.hotspot_overlap(d.contact_residues, d.hotspots),
    ) for d in designs])

spec = pd.concat([spec_df(bc, "bindcraft"), spec_df(rf, "rfdiffusion")], ignore_index=True)
spec.to_csv("results/specificity.csv", index=False)
print("wrote results/specificity.csv", spec.shape, " (SYNTHETIC mock numbers)")
print(spec[["design_id", "paradigm", "pae_fibril", "pae_monomer", "specificity_gap"]].head(6).to_string(index=False))

## 2 · Define "fibril-selective" and count it (per paradigm)

A design is **fibril-selective** if (a) it is a decent fibril binder (`pae_fibril ≤ 10`, the shared
binder cutoff) **and** (b) it clears a conformational-selectivity margin (`specificity_gap ≥ GAP_MIN`).
Choose `GAP_MIN` and **justify it** — there is no universal value; it trades selectivity stringency
against yield. Report how many designs survive BOTH conditions, per paradigm.

In [ ]:
PAE_FIBRIL_MAX = 10.0   # shared binder cutoff (good fibril binder)
GAP_MIN        = 4.0    # conformational-selectivity margin (JUSTIFY; tune in your report)

spec["good_fibril_binder"] = spec["pae_fibril"] <= PAE_FIBRIL_MAX
spec["fibril_selective"]   = spec["good_fibril_binder"] & (spec["specificity_gap"] >= GAP_MIN)

print(f"Selectivity definition: pae_fibril <= {PAE_FIBRIL_MAX} AND specificity_gap >= {GAP_MIN}")
for p, g in spec.groupby("paradigm"):
    nb = int(g["good_fibril_binder"].sum()); ns = int(g["fibril_selective"].sum()); n = len(g)
    print(f"  {p:12s}: good fibril binders {nb}/{n}; ALSO fibril-selective {ns}/{n} "
          f"({100*ns/max(n,1):.1f}%)  [SYNTHETIC if mock]")
print("\nExpect the selective fraction to be SMALL — rejecting the monomer is the hard requirement.")

In [ ]:
# Figure: specificity gap distribution + the selectivity quadrant (pae_fibril vs pae_monomer).
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for p, g in spec.groupby("paradigm"):
    ax[0].hist(g["specificity_gap"].dropna(), bins=15, alpha=0.5, label=p)
ax[0].axvline(GAP_MIN, color="k", ls="--", lw=1, label=f"GAP_MIN={GAP_MIN}")
ax[0].set_xlabel("specificity_gap = pae_monomer - pae_fibril (Å; >0 prefers fibril)")
ax[0].set_ylabel("designs"); ax[0].set_title("Conformational selectivity"); ax[0].legend()

# pae_fibril (x) vs pae_monomer (y): selective designs are LOW-x, HIGH-y (upper-left).
for p, g in spec.groupby("paradigm"):
    ax[1].scatter(g["pae_fibril"], g["pae_monomer"], s=14, alpha=0.6, label=p)
lim = [spec[["pae_fibril", "pae_monomer"]].min().min() - 1, spec[["pae_fibril", "pae_monomer"]].max().max() + 1]
ax[1].plot(lim, lim, "k--", lw=1, label="no selectivity (y=x)")
ax[1].axvline(PAE_FIBRIL_MAX, color="grey", ls=":", lw=1)
ax[1].set_xlabel("pae_fibril (Å, lower = better fibril binder)")
ax[1].set_ylabel("pae_monomer (Å, higher = rejects monomer)")
ax[1].set_title("Selectivity quadrant (want upper-left)"); ax[1].legend()
fig.suptitle("Conformational specificity — fibril vs monomer (EXAMPLE_DATA if mock)")
plt.tight_layout(); plt.savefig("results/p11_specificity.png", dpi=150); plt.show()
print("saved results/p11_specificity.png")

## 3 · Head-to-head: BindCraft vs RFdiffusion

Same fair comparison as the binder-family template: hit rate (here, the **fibril-selective** rate is
the meaningful one) and the `pae_interaction` / interface-energy distributions. Report the
*distribution*, not the single best. Mock numbers are SYNTHETIC.

In [ ]:
ranked = pd.read_csv("results/all_ranked.csv")
# attach specificity to the ranked survivors by design_id
spec_idx = spec.set_index("design_id")
ranked["specificity_gap"] = ranked["design_id"].map(spec_idx["specificity_gap"])
ranked["pae_monomer"]     = ranked["design_id"].map(spec_idx["pae_monomer"])
ranked["fibril_selective"]= ranked["design_id"].map(spec_idx["fibril_selective"]).fillna(False)

summary = []
for p, g in ranked.groupby("paradigm"):
    n = len(g)
    passed = int((g["layers_passed"] >= 3).sum())
    sel = int(g["fibril_selective"].sum())
    summary.append(dict(paradigm=p, n=n, all_layers_survivors=passed,
                        hit_rate_pct=round(100*passed/max(n,1), 1),
                        fibril_selective=sel, selective_pct=round(100*sel/max(n,1), 1),
                        median_pae=round(float(g["pae_interaction"].median()), 2),
                        median_gap=round(float(g["specificity_gap"].median()), 2)))
summary = pd.DataFrame(summary)
print("head-to-head summary (SYNTHETIC if mock):")
print(summary.to_string(index=False))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
for p, g in ranked.groupby("paradigm"):
    surv = g[g["layers_passed"] >= 3]
    ax[0].hist(g["pae_interaction"].dropna(), bins=15, alpha=0.5, label=p)
    ax[1].hist(surv["rosetta_dG"].dropna(), bins=15, alpha=0.5, label=p)
ax[0].set_xlabel("pae_interaction (Å, fibril; lower better)"); ax[0].set_ylabel("designs"); ax[0].set_title("AF2-Multimer pae_interaction"); ax[0].legend()
ax[1].set_xlabel("rosetta_dG (REU, more negative better)"); ax[1].set_title("Interface energy (survivors)"); ax[1].legend()
fig.suptitle("BindCraft vs RFdiffusion (EXAMPLE_DATA if mock)")
plt.tight_layout(); plt.savefig("results/p11_headtohead.png", dpi=150); plt.show()
print("saved results/p11_headtohead.png")

## 4 · Cross-amyloid specificity (tau vs α-syn) `[extension]`

A tau-fibril tracer should **not** light up α-synuclein fibrils (and vice versa) — otherwise it cannot
tell Alzheimer's from Parkinson's pathology. The cross-amyloid test scores each tau-designed binder
against the **α-synuclein** fibril as an off-target conformer. On Colab, build the α-syn fibril target
(6CU7/6H6B), re-run AF2-Multimer, and compute a **cross-amyloid gap** the same way. Here we scaffold
it with the mock backend (different `target` ⇒ different deterministic numbers) so the analysis shape
is in place.

In [ ]:
# Cross-amyloid counter-test: score the TAU-designed binders against the ALPHA-SYN fibril (off-target).
# On Colab: replace target="ASYN_FIBRIL" with the cleaned 6CU7/6H6B protofilament and run real AF2-Multimer.
def cross_amyloid_gap(designs, off_target="ASYN_FIBRIL", tool="mock"):
    rows = []
    for d in designs:
        on  = bt.af2_multimer(d.sequence, target=d.target,   tool=tool, hotspots=d.hotspots, conformer="fibril")
        off = bt.af2_multimer(d.sequence, target=off_target, tool=tool, hotspots=d.hotspots, conformer="fibril")
        rows.append(dict(design_id=d.design_id, paradigm=d.paradigm,
                         pae_on_target=on["pae_interaction"], pae_off_target=off["pae_interaction"],
                         cross_amyloid_gap=round(off["pae_interaction"] - on["pae_interaction"], 3)))
    return pd.DataFrame(rows)

cross = cross_amyloid_gap(bc + rf, off_target="ASYN_FIBRIL", tool="mock")
cross.to_csv("results/cross_amyloid.csv", index=False)
print("wrote results/cross_amyloid.csv", cross.shape, " (SYNTHETIC mock numbers)")
print("cross-amyloid gap = pae(off-target alpha-syn) - pae(on-target tau); >0 means tau-preferring.")
print(cross.groupby("paradigm")["cross_amyloid_gap"].median().round(2).to_dict(),
      " (median cross-amyloid gap per paradigm; SYNTHETIC)")

## 5 · Select the top conformation-selective candidates per paradigm

The D★ deliverable wants a **binder set** that is fibril-selective. Rank the fibril-selective survivors
by the fibril composite score, tie-break on a larger `specificity_gap`, and keep the top per paradigm.
Save the shortlist for the validation plan (notebook 05). If very few are selective, **say so** — that
is the honest, expected outcome.

In [ ]:
# Bring the epitope-coverage proxy onto the ranked survivors (it lives in the specificity table).
ranked["hotspot_overlap"] = ranked["design_id"].map(spec_idx["hotspot_overlap"])

top_per = []
for p, g in ranked.groupby("paradigm"):
    sel = g[(g["layers_passed"] >= 3) & (g["fibril_selective"] == True)]
    sel = sel.sort_values(["score", "specificity_gap"], ascending=False).head(20)
    top_per.append(sel)
top = pd.concat(top_per, ignore_index=True)
top.to_csv("results/top_candidates.csv", index=False)
print("wrote results/top_candidates.csv:", top.shape, "(top<=20 fibril-selective per paradigm)")
print("fibril-selective top-set per paradigm:", top.groupby("paradigm").size().to_dict())
if len(top) == 0:
    print("NOTE: zero fibril-selective survivors — a legitimate outcome. Loosen GAP_MIN OR report a 0% selective rate honestly.")
cols = [c for c in ["design_id", "paradigm", "score", "pae_interaction", "specificity_gap", "hotspot_overlap"] if c in top.columns]
top.head(8)[cols] if len(top) else "no fibril-selective candidates at this GAP_MIN"

## D3 (part 2) checklist
- [ ] Conformational-specificity test run for every design (`results/specificity.csv`): `pae_fibril`, `pae_monomer`, `specificity_gap`.
- [ ] "Fibril-selective" defined (`pae_fibril ≤ cutoff` AND `specificity_gap ≥ GAP_MIN`) and **justified**; selective fraction reported **per paradigm** (figure `results/p11_specificity.png`).
- [ ] Head-to-head: fibril hit rate + interface-energy distribution per paradigm (figure `results/p11_headtohead.png`).
- [ ] Cross-amyloid specificity (tau vs α-syn) computed or scaffolded (`results/cross_amyloid.csv`).
- [ ] `results/top_candidates.csv`: top fibril-selective set per paradigm (or an honest "few/none selective").
- [ ] Honest discussion: most designs fail the monomer counter-test; selectivity is a hypothesis until the assay.

**Next:** `05_validation_plan.ipynb` — the fibril-vs-monomer ELISA/SPR plan + the diagnostic-tracer framing.